# SIC Bias Shift And Clipping Diagnostic

This notebook tests the hypothesis that the bad sea-ice concentration calibration plot is partly caused by a simple additive bias and/or missing clipping to the physical interval `[0, 1]`.

It uses the saved tensor archives from a `synthetic_eval.cli --save-tensors` run and recomputes the concentration rank histogram after transforms of the form:

```python
sic_corrected = clip(sic + shift, 0, 1)
```

It also inspects how much probability mass is exactly or nearly at 0 and 1. This is useful because sea-ice concentration is often boundary-heavy, so rank histograms can be dominated by clipping, ties, and small shifts near the endpoints.


In [ ]:
from __future__ import annotations

import json
import math
import os
from collections import defaultdict
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from IPython.display import Image, Markdown, display

try:
    import pandas as pd
except ImportError:
    pd = None

from synthetic_eval.metrics import interval_coverage, rank_histogram, spread_skill, weighted_mean

plt.rcParams.update({
    "figure.dpi": 120,
    "axes.grid": True,
    "grid.alpha": 0.25,
})


## 1. Locate Saved Evaluation Output

Set `SYNTH_EVAL_OUT=/path/to/output` if the automatic search chooses the wrong run. The output directory must contain `samples/**/*.npz`, which are produced by `synthetic_eval.cli --save-tensors`.


In [ ]:
def existing_paths(*items):
    return [Path(x).expanduser() for x in items if x and Path(x).expanduser().exists()]


def default_data_roots():
    roots = []
    if os.environ.get("DATA_ROOT"):
        roots.append(Path(os.environ["DATA_ROOT"]).expanduser())
    roots.extend([
        Path("/Users/amir/sciml/sea_ice_data"),
        Path("/mnt/sciml/a.sadreev/sea_ice_data"),
        Path.cwd(),
    ])
    out = []
    seen = set()
    for root in roots:
        if root.exists() and root not in seen:
            out.append(root)
            seen.add(root)
    return out


def find_latest_synth_eval_dir():
    if os.environ.get("SYNTH_EVAL_OUT"):
        out = Path(os.environ["SYNTH_EVAL_OUT"]).expanduser()
        if out.exists():
            return out
        raise FileNotFoundError(f"SYNTH_EVAL_OUT does not exist: {out}")

    candidates = []
    for root in default_data_roots():
        candidates.extend([p for p in root.glob("synthetic_eval*") if p.is_dir()])
        candidates.extend([p for p in root.glob("*/synthetic_eval*") if p.is_dir()])
    candidates = [p for p in candidates if (p / "samples").exists() or (p / "arrays").exists()]
    if not candidates:
        roots = "\n".join(f"  - {p}" for p in default_data_roots())
        raise FileNotFoundError(
            "No synthetic_eval output directory found. Set SYNTH_EVAL_OUT to a run directory with saved tensors.\n"
            f"Searched roots:\n{roots}"
        )
    return sorted(candidates, key=lambda p: p.stat().st_mtime, reverse=True)[0]


OUT_DIR = find_latest_synth_eval_dir()
PLOTS_DIR = OUT_DIR / "plots"
ARRAYS_DIR = OUT_DIR / "arrays"
SAMPLES_DIR = OUT_DIR / "samples"

metadata_path = OUT_DIR / "metadata.json"
metadata = json.loads(metadata_path.read_text()) if metadata_path.exists() else {}

print("OUT_DIR    =", OUT_DIR)
print("PLOTS_DIR  =", PLOTS_DIR)
print("ARRAYS_DIR =", ARRAYS_DIR)
print("SAMPLES_DIR=", SAMPLES_DIR)
display(metadata)


## 2. Original Plots From The Run

This section shows the already saved CLI plots first, so the shift/clipping experiment can be compared against the original figures.


In [ ]:
def show_png(path: Path, width=950):
    if path.exists():
        display(Markdown(f"**{path.relative_to(OUT_DIR)}**"))
        display(Image(filename=str(path), width=width))
    else:
        print("missing:", path)


def show_many(pattern: str, width=950, limit=None):
    paths = sorted(PLOTS_DIR.glob(pattern))
    if limit is not None:
        paths = paths[:limit]
    if not paths:
        print(f"no plots matched {pattern}")
    for path in paths:
        show_png(path, width=width)


display(Markdown("### Original rank histograms"))
show_many("rank_hist_*concentration*.png", width=900)

display(Markdown("### Original example panels"))
show_many("example_*.png", width=1050, limit=6)


## 3. Load Tensor Archives

Use `CONDITION_FILTER` to restrict the analysis to one condition directory such as `swath_r00_d0p05_n0`.


In [ ]:
CONDITION_FILTER = ""  # substring in parent directory name; leave empty for all conditions
MAX_ARCHIVES = None    # set to a small integer for quick iteration, e.g. 20
RANK_STRIDE = int(metadata.get("rank_stride", 4) or 4)
EVAL_REGION = metadata.get("eval_region", "unobserved")
CHANNEL = 0  # concentration
LEVELS = (0.5, 0.8, 0.9, 0.95)

all_tensor_paths = sorted(SAMPLES_DIR.glob("**/*.npz"))
if CONDITION_FILTER:
    all_tensor_paths = [p for p in all_tensor_paths if CONDITION_FILTER in p.parent.name]
if MAX_ARCHIVES is not None:
    all_tensor_paths = all_tensor_paths[: int(MAX_ARCHIVES)]

if not all_tensor_paths:
    raise FileNotFoundError(
        f"No saved tensor archives found under {SAMPLES_DIR}. "
        "Rerun synthetic_eval.cli with --save-tensors or set SYNTH_EVAL_OUT."
    )

print("n tensor archives =", len(all_tensor_paths))
for path in all_tensor_paths[:10]:
    print(path.relative_to(OUT_DIR))
if len(all_tensor_paths) > 10:
    print(f"... {len(all_tensor_paths) - 10} more")


In [ ]:
def scalar_str(value):
    arr = np.asarray(value)
    return str(arr.item() if arr.shape == () else arr)


def archive_condition_tag(path: Path, z=None):
    if z is not None and "condition_tag" in z.files:
        return scalar_str(z["condition_tag"])
    return path.parent.name


def load_valid_mask(shape_hw):
    candidates = []
    if metadata.get("mask_path"):
        candidates.append(Path(metadata["mask_path"]))
    if metadata.get("data_root"):
        candidates.append(Path(metadata["data_root"]) / "mask_padding.npy")
    for root in default_data_roots():
        candidates.append(root / "mask_padding.npy")
    for path in candidates:
        if path.exists():
            mask = np.load(path).astype(np.float32)
            if mask.ndim == 3:
                mask = mask[0]
            if tuple(mask.shape) == tuple(shape_hw):
                return mask > 0
    return np.ones(shape_hw, dtype=bool)


def evaluation_mask(observed_mask, valid_mask, eval_region):
    observed = np.asarray(observed_mask) > 0
    valid = np.asarray(valid_mask) > 0
    if eval_region == "all":
        return valid
    if eval_region == "observed":
        return valid & observed
    if eval_region == "unobserved":
        return valid & ~observed
    raise ValueError(f"Unknown eval_region: {eval_region}")


with np.load(all_tensor_paths[0], allow_pickle=False) as z0:
    sample_ensemble = z0["ensemble"]
    sample_truth = z0["truth"]
    sample_mask = z0["mask"]
    valid_mask = load_valid_mask(sample_truth.shape[-2:])
    print("sample archive:", all_tensor_paths[0].relative_to(OUT_DIR))
    print("ensemble shape:", sample_ensemble.shape)
    print("truth shape:   ", sample_truth.shape)
    print("mask shape:    ", sample_mask.shape)
    print("valid fraction:", float(valid_mask.mean()))
    print("eval region:   ", EVAL_REGION)
    print("rank stride:   ", RANK_STRIDE)


## 4. SIC Value Diagnostics

This checks whether truth and ensemble concentration are actually exact boundary values, near boundary values, or continuous values inside the interval.


In [ ]:
def describe_values(values, prefix):
    values = np.asarray(values, dtype=np.float64).reshape(-1)
    finite = values[np.isfinite(values)]
    if finite.size == 0:
        return {f"{prefix}_n": 0}
    return {
        f"{prefix}_n": int(finite.size),
        f"{prefix}_min": float(finite.min()),
        f"{prefix}_p01": float(np.quantile(finite, 0.01)),
        f"{prefix}_p05": float(np.quantile(finite, 0.05)),
        f"{prefix}_mean": float(finite.mean()),
        f"{prefix}_p50": float(np.quantile(finite, 0.50)),
        f"{prefix}_p95": float(np.quantile(finite, 0.95)),
        f"{prefix}_p99": float(np.quantile(finite, 0.99)),
        f"{prefix}_max": float(finite.max()),
        f"{prefix}_exact0": float(np.mean(finite == 0.0)),
        f"{prefix}_exact1": float(np.mean(finite == 1.0)),
        f"{prefix}_near0_1e-3": float(np.mean(finite <= 1e-3)),
        f"{prefix}_near1_1e-3": float(np.mean(finite >= 1.0 - 1e-3)),
        f"{prefix}_near0_1e-2": float(np.mean(finite <= 1e-2)),
        f"{prefix}_near1_1e-2": float(np.mean(finite >= 1.0 - 1e-2)),
        f"{prefix}_outside_0_1": float(np.mean((finite < 0.0) | (finite > 1.0))),
    }


def sample_values_for_hist(paths, max_archives=24, max_values_per_archive=30000, seed=123):
    rng = np.random.default_rng(seed)
    ens_values = []
    truth_values = []
    residual_values = []
    selected = paths[: min(len(paths), max_archives)]
    for path in selected:
        with np.load(path, allow_pickle=False) as z:
            ens = z["ensemble"][:, CHANNEL]
            truth = z["truth"][CHANNEL]
            mask = evaluation_mask(z["mask"], valid_mask, EVAL_REGION)
            flat_mask = mask.reshape(-1)
            ens_flat = ens.reshape(ens.shape[0], -1)[:, flat_mask].reshape(-1)
            truth_flat = truth.reshape(-1)[flat_mask]
            mean_flat = ens.mean(axis=0).reshape(-1)[flat_mask]
            if ens_flat.size > max_values_per_archive:
                ens_flat = rng.choice(ens_flat, size=max_values_per_archive, replace=False)
            if truth_flat.size > max_values_per_archive:
                idx = rng.choice(truth_flat.size, size=max_values_per_archive, replace=False)
                truth_flat = truth_flat[idx]
                mean_for_resid = mean_flat[idx]
            else:
                mean_for_resid = mean_flat
            ens_values.append(ens_flat)
            truth_values.append(truth_flat)
            residual_values.append(mean_for_resid - truth_flat)
    return np.concatenate(ens_values), np.concatenate(truth_values), np.concatenate(residual_values)


ens_vals, truth_vals, residual_vals = sample_values_for_hist(all_tensor_paths)
summary = {
    **describe_values(truth_vals, "truth"),
    **describe_values(ens_vals, "ensemble"),
    **describe_values(residual_vals, "mean_minus_truth"),
}
if pd is not None:
    display(pd.DataFrame([summary]).T.rename(columns={0: "value"}))
else:
    display(summary)


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].hist(truth_vals, bins=np.linspace(-0.05, 1.05, 80), alpha=0.65, label="truth", density=True)
axes[0].hist(ens_vals, bins=np.linspace(-0.05, 1.05, 80), alpha=0.55, label="ensemble", density=True)
axes[0].set_title("SIC values")
axes[0].set_xlabel("concentration")
axes[0].set_ylabel("density")
axes[0].legend()

axes[1].hist(ens_vals - np.clip(ens_vals, 0.0, 1.0), bins=80, color="tab:orange")
axes[1].set_title("Raw ensemble clipping correction")
axes[1].set_xlabel("raw - clip(raw, 0, 1)")

axes[2].hist(residual_vals, bins=80, color="tab:red")
axes[2].axvline(0.0, color="black", linewidth=1)
axes[2].set_title("Ensemble mean residual")
axes[2].set_xlabel("mean(SIC ensemble) - truth")

plt.tight_layout()
plt.show()


## 5. Shift And Clipping Grid

`SHIFT_VALUES` is intentionally wider than a tiny numerical correction. If the best point is far from zero, the rank histogram tilt is likely a real systematic bias. If clipping alone or a tiny shift fixes it, the issue is probably a physical-range postprocessing mismatch.


In [ ]:
SHIFT_VALUES = np.array([
    -0.25, -0.20, -0.15, -0.10, -0.075, -0.05, -0.025,
     0.0,
     0.025, 0.05, 0.075, 0.10, 0.15, 0.20, 0.25,
], dtype=np.float64)
CLIP_OPTIONS = [False, True]
EDGE_SNAP_EPS_VALUES = [None]  # add e.g. 0.01 or 0.05 to test force-to-0/1 after clipping


def corrected_sic(ens_sic, shift=0.0, clip=False, edge_snap_eps=None):
    out = np.asarray(ens_sic, dtype=np.float64) + float(shift)
    if clip:
        out = np.clip(out, 0.0, 1.0)
    if edge_snap_eps is not None:
        eps = float(edge_snap_eps)
        out = out.copy()
        out[out <= eps] = 0.0
        out[out >= 1.0 - eps] = 1.0
    return out


def transform_label(shift, clip, edge_snap_eps=None):
    base = f"shift={shift:+.3f}"
    if clip:
        base += ", clip"
    else:
        base += ", no clip"
    if edge_snap_eps is not None:
        base += f", snap={edge_snap_eps:g}"
    return base


def rank_shape_scores(counts):
    counts = np.asarray(counts, dtype=np.float64)
    total = counts.sum()
    if total <= 0:
        return {
            "total_ranks": 0,
            "l1_flat": np.nan,
            "edge_ratio_vs_flat": np.nan,
            "center_ratio_vs_flat": np.nan,
            "right_minus_left_edges_in_flat_units": np.nan,
            "chi2_like_not_independent": np.nan,
        }
    probs = counts / total
    expected = 1.0 / len(probs)
    center_slice = probs[len(probs) // 3 : 2 * len(probs) // 3]
    return {
        "total_ranks": int(total),
        "l1_flat": float(np.abs(probs - expected).sum()),
        "edge_ratio_vs_flat": float((probs[0] + probs[-1]) / (2 * expected)),
        "center_ratio_vs_flat": float(center_slice.mean() / expected),
        "right_minus_left_edges_in_flat_units": float((probs[-1] - probs[0]) / expected),
        "chi2_like_not_independent": float(((counts - total * expected) ** 2 / max(total * expected, 1.0)).sum()),
    }


def evaluate_transform(paths, shift=0.0, clip=False, edge_snap_eps=None, rank_stride=4):
    counts_total = None
    metric_rows = []
    by_condition_counts = {}
    for archive_i, path in enumerate(paths):
        with np.load(path, allow_pickle=False) as z:
            ens = z["ensemble"][:, CHANNEL]
            truth = z["truth"][CHANNEL].astype(np.float64)
            obs_mask = z["mask"]
            condition_tag = archive_condition_tag(path, z)
            combo_seed = int(np.asarray(z["combo_seed"]).item()) if "combo_seed" in z.files else archive_i

        mask = evaluation_mask(obs_mask, valid_mask, EVAL_REGION)
        ens_corr = corrected_sic(ens, shift=shift, clip=clip, edge_snap_eps=edge_snap_eps)
        mean = ens_corr.mean(axis=0)
        diff = mean - truth

        rank_mask = mask[::rank_stride, ::rank_stride]
        counts = rank_histogram(
            ens_corr[:, ::rank_stride, ::rank_stride],
            truth[::rank_stride, ::rank_stride],
            seed=combo_seed + CHANNEL,
            weights_mask=rank_mask,
        )
        counts_total = counts.copy() if counts_total is None else counts_total + counts
        by_condition_counts[condition_tag] = by_condition_counts.get(condition_tag, np.zeros_like(counts)) + counts

        cover = interval_coverage(ens_corr, truth, LEVELS)
        ss = spread_skill(ens_corr, truth, weights=mask)
        metric_rows.append({
            "rmse": math.sqrt(weighted_mean(diff ** 2, mask)),
            "mean_error": weighted_mean(diff, mask),
            "mean_abs_error": weighted_mean(np.abs(diff), mask),
            "coverage_0.5": weighted_mean(cover["0.5"].astype(np.float64), mask),
            "coverage_0.8": weighted_mean(cover["0.8"].astype(np.float64), mask),
            "coverage_0.9": weighted_mean(cover["0.9"].astype(np.float64), mask),
            "coverage_0.95": weighted_mean(cover["0.95"].astype(np.float64), mask),
            **ss,
        })

    aggregate = {key: float(np.nanmean([row[key] for row in metric_rows])) for key in metric_rows[0].keys()}
    return counts_total, aggregate, by_condition_counts


grid_rows = []
counts_by_label = {}
for edge_snap_eps in EDGE_SNAP_EPS_VALUES:
    for clip in CLIP_OPTIONS:
        for shift in SHIFT_VALUES:
            counts, metrics, _ = evaluate_transform(
                all_tensor_paths,
                shift=float(shift),
                clip=bool(clip),
                edge_snap_eps=edge_snap_eps,
                rank_stride=RANK_STRIDE,
            )
            label = transform_label(float(shift), bool(clip), edge_snap_eps)
            counts_by_label[label] = counts
            grid_rows.append({
                "label": label,
                "shift": float(shift),
                "clip": bool(clip),
                "edge_snap_eps": edge_snap_eps,
                **rank_shape_scores(counts),
                **metrics,
            })

grid = pd.DataFrame(grid_rows) if pd is not None else grid_rows
if pd is not None:
    display(grid.sort_values(["l1_flat", "rmse"]).head(20))
else:
    display(sorted(grid, key=lambda r: (r["l1_flat"], r["rmse"]))[:20])


In [ ]:
if pd is not None:
    fig, axes = plt.subplots(2, 2, figsize=(13, 9), sharex=True)
    for clip, sub in grid.groupby("clip"):
        suffix = "clip" if clip else "no clip"
        sub = sub[sub["edge_snap_eps"].isna()] if "edge_snap_eps" in sub else sub
        axes[0, 0].plot(sub["shift"], sub["l1_flat"], marker="o", label=suffix)
        axes[0, 1].plot(sub["shift"], sub["right_minus_left_edges_in_flat_units"], marker="o", label=suffix)
        axes[1, 0].plot(sub["shift"], sub["rmse"], marker="o", label=suffix)
        axes[1, 1].plot(sub["shift"], sub["mean_error"], marker="o", label=suffix)

    axes[0, 0].set_title("Rank histogram distance from flat")
    axes[0, 0].set_ylabel("L1 distance")
    axes[0, 1].set_title("Rank left/right tilt")
    axes[0, 1].axhline(0.0, color="black", linewidth=1)
    axes[0, 1].set_ylabel("right edge - left edge, flat units")
    axes[1, 0].set_title("RMSE of ensemble mean")
    axes[1, 0].set_xlabel("additive shift")
    axes[1, 0].set_ylabel("RMSE")
    axes[1, 1].set_title("Mean bias")
    axes[1, 1].axhline(0.0, color="black", linewidth=1)
    axes[1, 1].set_xlabel("additive shift")
    axes[1, 1].set_ylabel("mean ensemble - truth")
    for ax in axes.ravel():
        ax.legend()
    plt.tight_layout()
    plt.show()
else:
    print("Install pandas to draw grid curves conveniently.")


## 6. Rank Histograms For Raw, Clipped, And Best Corrections

The best correction here is selected by rank histogram flatness (`l1_flat`). Check RMSE and mean bias too, because a flatter rank histogram alone can be misleading for boundary-heavy variables.


In [ ]:
CORRECTED_PLOTS_DIR = PLOTS_DIR / "sic_bias_shift_clipping"
CORRECTED_PLOTS_DIR.mkdir(parents=True, exist_ok=True)


def safe_name(text):
    return (
        str(text)
        .replace("+", "plus")
        .replace("-", "minus")
        .replace(".", "p")
        .replace(", ", "__")
        .replace(" ", "_")
        .replace("=", "")
    )


def plot_one_rank_histogram(label, counts, path=None):
    counts = np.asarray(counts, dtype=np.float64)
    probs = counts / max(1.0, counts.sum())
    expected = 1.0 / len(probs)
    scores = rank_shape_scores(counts)
    fig, ax = plt.subplots(figsize=(8, 4))
    ax.bar(np.arange(len(probs)), probs, color="black", width=0.85)
    ax.axhline(expected, color="tab:red", linestyle="--", linewidth=1.2, label="flat")
    ax.set_title(f"{label}\nL1={scores['l1_flat']:.3f}, bias={scores['right_minus_left_edges_in_flat_units']:.3f}")
    ax.set_xlabel("rank of truth")
    ax.set_ylabel("probability")
    ax.legend()
    fig.tight_layout()
    if path is not None:
        fig.savefig(path, dpi=160)
        print("saved:", path)
    plt.show()


def plot_rank_histograms(labels, counts_lookup, ncols=3, path=None):
    n = len(labels)
    ncols = min(ncols, n)
    nrows = int(math.ceil(n / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(5.0 * ncols, 3.5 * nrows), squeeze=False)
    for ax, label in zip(axes.ravel(), labels):
        counts = counts_lookup[label].astype(np.float64)
        probs = counts / max(1.0, counts.sum())
        expected = 1.0 / len(probs)
        ax.bar(np.arange(len(probs)), probs, color="black", width=0.85)
        ax.axhline(expected, color="tab:red", linestyle="--", linewidth=1.2)
        ax.set_title(label)
        ax.set_xlabel("rank of truth")
        ax.set_ylabel("probability")
    for ax in axes.ravel()[len(labels):]:
        ax.axis("off")
    plt.tight_layout()
    if path is not None:
        fig.savefig(path, dpi=160)
        print("saved:", path)
    plt.show()


if pd is not None:
    raw_label = transform_label(0.0, False, None)
    clipped_label = transform_label(0.0, True, None)
    top_labels = list(grid.sort_values(["l1_flat", "rmse"])["label"].head(7))
    labels = []
    for label in [raw_label, clipped_label, *top_labels]:
        if label not in labels:
            labels.append(label)
    best_row = grid.sort_values(["l1_flat", "rmse"]).iloc[0]
    best_label = str(best_row["label"])
    print("best corrected rank histogram:", best_label)
    display(best_row.to_frame().T)

    comparison_path = CORRECTED_PLOTS_DIR / "rank_histograms_raw_clipped_best.png"
    plot_rank_histograms(labels, counts_by_label, ncols=3, path=comparison_path)

    corrected_png = CORRECTED_PLOTS_DIR / f"rank_hist_corrected__{safe_name(best_label)}.png"
    plot_one_rank_histogram(best_label, counts_by_label[best_label], path=corrected_png)

    counts_npz = CORRECTED_PLOTS_DIR / "rank_histogram_counts_raw_clipped_best.npz"
    np.savez_compressed(
        counts_npz,
        **{safe_name(label): np.asarray(counts_by_label[label], dtype=np.int64) for label in labels},
    )
    print("saved:", counts_npz)
else:
    print("Install pandas to select top labels automatically.")


## 7. Best Transform Per Condition

If different swath repeats or densities want different shifts, the issue may be condition-dependent rather than one global bias.


In [ ]:
condition_groups = defaultdict(list)
for path in all_tensor_paths:
    condition_groups[path.parent.name].append(path)

condition_rows = []
for condition_tag, paths in sorted(condition_groups.items()):
    best = None
    for clip in CLIP_OPTIONS:
        for shift in SHIFT_VALUES:
            counts, metrics, _ = evaluate_transform(paths, shift=float(shift), clip=bool(clip), rank_stride=RANK_STRIDE)
            row = {
                "condition_tag": condition_tag,
                "n_archives": len(paths),
                "label": transform_label(float(shift), bool(clip), None),
                "shift": float(shift),
                "clip": bool(clip),
                **rank_shape_scores(counts),
                **metrics,
            }
            if best is None or (row["l1_flat"], row["rmse"]) < (best["l1_flat"], best["rmse"]):
                best = row
    condition_rows.append(best)

if pd is not None:
    condition_best = pd.DataFrame(condition_rows).sort_values("condition_tag")
    display(condition_best)
else:
    display(condition_rows)


## 8. Visual Case Comparison

Set `ARCHIVE_INDEX` and use the globally best correction to inspect the spatial effect on one case.


In [ ]:
ARCHIVE_INDEX = 0

if pd is not None:
    best_row = grid.sort_values(["l1_flat", "rmse"]).iloc[0]
    BEST_SHIFT = float(best_row["shift"])
    BEST_CLIP = bool(best_row["clip"])
else:
    BEST_SHIFT = 0.0
    BEST_CLIP = True
BEST_SNAP_EPS = None

path = all_tensor_paths[min(ARCHIVE_INDEX, len(all_tensor_paths) - 1)]
with np.load(path, allow_pickle=False) as z:
    ens_raw = z["ensemble"][:, CHANNEL]
    truth = z["truth"][CHANNEL]
    obs_mask = z["mask"]
    observed = z["observed"][CHANNEL]
    condition_tag = archive_condition_tag(path, z)

ens_corr = corrected_sic(ens_raw, BEST_SHIFT, BEST_CLIP, BEST_SNAP_EPS)
raw_mean = ens_raw.mean(axis=0)
corr_mean = ens_corr.mean(axis=0)
raw_spread = ens_raw.std(axis=0)
corr_spread = ens_corr.std(axis=0)
mask = evaluation_mask(obs_mask, valid_mask, EVAL_REGION)

print("archive:", path.relative_to(OUT_DIR))
print("condition:", condition_tag)
print("best correction:", transform_label(BEST_SHIFT, BEST_CLIP, BEST_SNAP_EPS))

fig, axes = plt.subplots(2, 4, figsize=(16, 7))
panels = [
    (truth, "truth", "viridis", 0.0, 1.0),
    (observed, "observed", "viridis", 0.0, 1.0),
    (raw_mean, "raw mean", "viridis", 0.0, 1.0),
    (corr_mean, "corrected mean", "viridis", 0.0, 1.0),
    (obs_mask, "observed mask", "gray", None, None),
    (mask.astype(float), f"eval mask: {EVAL_REGION}", "gray", None, None),
    (raw_mean - truth, "raw mean - truth", "coolwarm", -0.5, 0.5),
    (corr_mean - truth, "corrected mean - truth", "coolwarm", -0.5, 0.5),
]
for ax, (arr, title, cmap, vmin, vmax) in zip(axes.ravel(), panels):
    im = ax.imshow(arr, cmap=cmap, vmin=vmin, vmax=vmax)
    ax.set_title(title)
    ax.axis("off")
    plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
plt.tight_layout()
plt.show()

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].hist(raw_spread[mask], bins=60, alpha=0.75, label="raw")
axes[0].hist(corr_spread[mask], bins=60, alpha=0.55, label="corrected")
axes[0].set_title("Spread distribution")
axes[0].set_xlabel("SIC std")
axes[0].legend()
axes[1].hist((raw_mean - truth)[mask], bins=60, alpha=0.75, label="raw")
axes[1].hist((corr_mean - truth)[mask], bins=60, alpha=0.55, label="corrected")
axes[1].axvline(0.0, color="black", linewidth=1)
axes[1].set_title("Mean residual distribution")
axes[1].set_xlabel("mean - truth")
axes[1].legend()
plt.tight_layout()
plt.show()


## 9. Interpretation Checklist

Use this as a quick readout after running the notebook:

- If `clip=True, shift=0` is much better than raw, the original diagnostic likely mixed in out-of-range SIC values.
- If the best `shift` is small and mean bias moves close to zero, a simple additive postprocessing correction is plausible.
- If rank flatness improves only with a large shift while RMSE/coverage get worse, the rank histogram is being gamed rather than fixed.
- If the best shifts differ strongly by condition, inspect masks/densities separately.
- If truth and ensemble have many exact ties at 0 or 1, random tie-breaking and endpoint clipping can dominate the rank histogram; compare the value histograms before trusting a single calibration plot.
